# Session 3 — Analysis: Model Depth and Memory Limits

## Background

Sessions 1–2 measured a single Transformer block. This session stacks blocks to BERT-scale (12) and GPT-2-scale (24) and finds where each device runs out of memory. The chart of throughput vs depth — with the GPU line ending abruptly where the TPU line continues — is the most memorable visual in the series.

## Goal

- Identify the OOM boundary on each device at `batch=64`
- Compare how throughput scales with depth before the OOM boundary
- Estimate usable model depth from VRAM figures (GPU only)

## Question

At `batch=64`, what is each device's maximum usable depth, and how does throughput scale with depth up to that limit?

---

**Prerequisites:** Run `01-benchmark-gpu.ipynb` and `02-benchmark-tpu.ipynb` first.

In [1]:
import json, os

def load_json(path):
    if not os.path.exists(path):
        print(f"  Missing: {path}")
        return None
    with open(path) as f:
        return json.load(f)

gpu = load_json("results/gpu_depth.json")
tpu = load_json("results/tpu_depth.json")

N_LAYERS_SWEEP = [1, 2, 4, 6, 8, 12, 16, 24]

for label, d in [("GPU", gpu), ("TPU", tpu)]:
    if d:
        oom = [int(k) for k, v in d["results"].items() if v == "OOM"]
        ok  = [int(k) for k, v in d["results"].items() if v != "OOM"]
        print(f"  {label} ({d['device_name']}) — OK layers: {ok}  OOM layers: {oom}")
    else:
        print(f"  {label}: NOT LOADED")

  GPU (NVIDIA L4) — OK layers: [1, 2, 4, 6, 8, 12, 16, 24]  OOM layers: []
  TPU (TPU) — OK layers: [1, 2, 4, 6, 8, 12, 16, 24]  OOM layers: []


In [2]:
def get_tput(d, nl):
    if d is None: return None
    r = d["results"].get(str(nl))
    if r == "OOM" or r is None: return None
    return r["throughput"]

def get_vram(d, nl):
    if d is None: return None
    r = d["results"].get(str(nl))
    if r == "OOM" or r is None: return None
    return r.get("peak_vram_mb")

def is_oom(d, nl):
    if d is None: return False
    return d["results"].get(str(nl)) == "OOM"

print(f"  {'n_layers':<10}  {'GPU samples/s':>14}  {'GPU VRAM MB':>12}  {'TPU samples/s':>14}  {'TPU/GPU':>8}")
print(f"  {'─'*10}  {'─'*14}  {'─'*12}  {'─'*14}  {'─'*8}")
for nl in N_LAYERS_SWEEP:
    gt  = get_tput(gpu, nl)
    gv  = get_vram(gpu, nl)
    tt  = get_tput(tpu, nl)
    g_s = f"{gt:>14,.0f}" if gt else ("         OOM" if is_oom(gpu, nl) else "             —")
    gv_s= f"{gv:>12,.0f}" if gv else ("     OOM" if is_oom(gpu, nl) else "            —")
    t_s = f"{tt:>14,.0f}" if tt else ("         OOM" if is_oom(tpu, nl) else "             —")
    ratio = f"{tt/gt:>8.2f}×" if gt and tt else "       —"
    print(f"  {nl:<10}  {g_s}  {gv_s}  {t_s}  {ratio}")

  n_layers     GPU samples/s   GPU VRAM MB   TPU samples/s   TPU/GPU
  ──────────  ──────────────  ────────────  ──────────────  ────────
  1                    2,635           531           6,033      2.29×
  2                    1,244           890           3,106      2.50×
  4                      599         1,606           1,587      2.65×
  6                      393         2,323           1,059      2.70×
  8                      294         3,039             786      2.67×
  12                     199         4,473             519      2.61×
  16                     152         5,891             386      2.54×
  24                     101         8,772             254      2.52×


In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

# ── Chart 1: Throughput vs n_layers with OOM boundary ────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))

for d, color, label, marker in [
    (gpu, "#76b900", "GPU (NVIDIA L4)",    "o"),
    (tpu, "#4285F4", "TPU (v5litepod-1)",  "s"),
]:
    if d is None:
        continue
    xs, ys = [], []
    oom_x  = None
    for nl in N_LAYERS_SWEEP:
        t = get_tput(d, nl)
        if t is not None:
            xs.append(nl)
            ys.append(t)
        elif is_oom(d, nl) and oom_x is None:
            oom_x = nl
    if xs:
        ax.plot(xs, ys, marker=marker, label=label, color=color, linewidth=2, markersize=6)
    if oom_x is not None:
        ax.axvline(oom_x, color=color, linestyle=":", linewidth=1.5,
                   label=f"{label.split()[0]} OOM at n_layers={oom_x}")

ax.set_xlabel("Number of Transformer layers (n_layers)", fontsize=11)
ax.set_ylabel("Throughput (samples / second)", fontsize=11)
ax.set_title(f"Throughput vs Depth — batch={gpu['batch_size'] if gpu else 64}, seq_len=128", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, linestyle="--", alpha=0.4)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()

os.makedirs("../docs/assets", exist_ok=True)
plt.savefig("../docs/assets/session_6_chart_depth.png", dpi=150)
print("Saved session_6_chart_depth.png")

Saved session_6_chart_depth.png


In [4]:
# ── Chart 2: GPU VRAM vs depth ────────────────────────────────────────────────
if gpu:
    fig2, ax2 = plt.subplots(figsize=(8, 4))

    xs_vram = [nl for nl in N_LAYERS_SWEEP if get_vram(gpu, nl) is not None]
    ys_vram = [get_vram(gpu, nl) for nl in xs_vram]

    ax2.bar(range(len(xs_vram)), ys_vram, color="#76b900", alpha=0.8)
    ax2.set_xticks(range(len(xs_vram)))
    ax2.set_xticklabels(xs_vram)
    ax2.axhline(23700, color="red", linestyle="--", linewidth=1.5, label="L4 VRAM limit (23.7 GB)")
    ax2.set_xlabel("Number of Transformer layers", fontsize=11)
    ax2.set_ylabel("Peak VRAM (MB)", fontsize=11)
    ax2.set_title("GPU Peak VRAM by Depth (batch=64, seq_len=128)", fontsize=12)
    ax2.legend(fontsize=10)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout()
    plt.savefig("../docs/assets/session_6_chart_vram.png", dpi=150)
    print("Saved session_6_chart_vram.png")
else:
    print("GPU data not loaded — skipping VRAM chart.")

Saved session_6_chart_vram.png


## Interpreting the results

### Throughput vs depth

Each additional Transformer layer adds exactly the same dense-matmul workload. On the TPU, this maps perfectly to additional MXU utilisation — throughput is expected to decline approximately linearly with depth (more compute per step, same samples per step). On the GPU, the same decline occurs but OOM cuts it short.

### OOM boundary

The GPU OOM boundary depends on:
- **Activation memory** — every intermediate tensor in the forward pass is retained for backprop. Scales as `n_layers × batch × seq_len × d_model × 4 bytes` plus the attention matrices.
- **Optimizer state** — Adam stores two moment tensors per parameter. Scales as `2 × n_params × 4 bytes`.
- **Gradients** — same size as parameters.

At `batch=64, seq_len=128`, the dominant memory cost is activations. The OOM layer count observed here is the empirical answer for this configuration.

### TPU memory architecture

The TPU v5litepod-1 has 16 GB HBM2 — less total capacity than the L4's 23.7 GB, but HBM2 has ~2.7× the bandwidth. Depending on the activation checkpointing implementation and XLA's buffer allocation strategy, the TPU may reach OOM at a similar or slightly different layer count. If the TPU's OOM boundary is shallower (due to less raw capacity), this is the counter-argument to the throughput advantage shown in Sessions 1 and 4.

### Practical guidance

- Use the VRAM chart as a budget planner: if `peak_vram_mb` at the target n_layers exceeds your device's VRAM, switch to gradient checkpointing (`torch.utils.checkpoint`) or mixed precision (Session 4) before buying more hardware.
- A 12-layer BERT-scale model at `batch=64` is the reference point. If it fits on GPU, Sessions 6 and 7 together give you the throughput and break-even information needed to decide whether to move to TPU.